# QR Phishing Detection — URL Model Training (Colab)

Trains a **4-class** character-level Transformer that classifies a **URL** as
**benign / phishing / malware / defacement**. A QR code is just a container for a URL, so the
signal is in the URL text: the app scans the QR, decodes it to a URL, and this model judges it.

**Before running:** download `malicious_phish.csv` from
https://www.kaggle.com/datasets/sid321axn/malicious-urls-dataset and put it in your Google Drive
at `/content/drive/MyDrive/malicious_phish.csv`. Then: **Runtime → Run all**.

## How it works — phishing detection logic & training

This notebook builds the machine-learning model that judges a QR code's link. It is a
**4-class** classifier — for any URL it predicts one of:

| Class (label) | Meaning |
|---|---|
| `benign` (0) | safe / legitimate |
| `phishing` (1) | fake login / payment page (incl. typosquats, combosquats, subdomain abuse) |
| `malware` (2) | link that drops malicious software |
| `defacement` (3) | hacked / defaced page |

### 1. The problem ("Quishing")
Attackers hide a **bad URL** inside a QR code. People scan it without thinking and land on a
fake/harmful page. We must judge the link **before** the user opens it.

### 2. Why we analyze the URL (not the QR picture)
A QR code is just a **lossless encoding of text** — a safe QR and a bad QR look identical. The
danger is in the **URL inside**. So: `QR image → decode → URL text → ML model → class`.

### 3. How the model learns (supervised, no hand-written rules)
It is shown thousands of example URLs already labelled with one of the 4 classes, e.g.:

| Example URL | Label |
|---|---|
| `https://github.com/login` | benign |
| `http://paypal-verify.tk/login` | phishing |
| `http://evil.site/setup.exe` | malware |
| `http://hacked.example.com/x.html` | defacement |

From these it **learns the statistical patterns itself** (bad TLDs, `@` trick, IPs, look-alike
brands, keywords). For any **new** URL it outputs a probability for each class and picks the
highest (argmax). Sections 6–7 prove it works on **unseen** data + brand-new URLs.

### 4. The pipeline in this notebook
| Section | What it does |
|---|---|
| 1 | Mount Google Drive |
| 2 | Build the dataset: all 4 classes from the CSV + real legit domains (labels 0–3) |
| 2.5 | Attack augmentation (typo/combo/subdomain) + save one combined CSV |
| 3 | Balance the 4 classes + split train / val / test |
| 4 | Normalize each URL + tokenize characters |
| 5 | Build the Transformer (softmax over 4 classes) and **train** |
| 6 | Evaluate: accuracy, per-class report, 4×4 confusion matrix |
| 7 | Generalization + held-out brand test |
| 8 | Export the model + tokenizer (with class names) for the backend |

## SECTION 1: Mount Google Drive

Gives the notebook access to the dataset stored in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## SECTION 2: Build the dataset (4 classes + real, recent threat feeds)

Sources, for robustness and worldwide coverage:
1. **`malicious_phish.csv`** — labelled benign / phishing / **malware** / **defacement** URLs.
2. **Live threat feeds** (fresh, not just 2021): **OpenPhish** → phishing, **URLhaus** → malware.
   So the model sees *current* attack patterns, not only the static dataset. (Skipped gracefully
   if a feed is offline; the saved CSV then freezes whatever was fetched, keeping runs reproducible.)
3. **Top global domains** (Cisco Umbrella top‑1M) — popular legitimate sites worldwide → benign.
4. **~9,000 university domains** — teaches the `.edu` / `.edu.bd` / `.ac.*` pattern → benign.

Section 2.5 then adds synthetic brand attacks. All of it is saved to one combined CSV on Drive.

In [ ]:
import pandas as pd
import io, json, os, zipfile, urllib.request

# --- 4-CLASS setup -----------------------------------------------------------
CLASSES = ['benign', 'phishing', 'malware', 'defacement']   # labels 0, 1, 2, 3

# --- One combined dataset cached on your Drive -------------------------------
# v3 = adds strong brand-benign coverage (fixes false positives on real brands like
# paypal.com / sonalibank.com.bd). New filename => auto-rebuilds on Run all.
DATASET_CSV = '/content/drive/MyDrive/qr_training_dataset_v3.csv'
REBUILD_DATASET = False

CSV_PATH = '/content/drive/MyDrive/malicious_phish.csv'   # original source (read only when building)
TOP_DOMAINS_N = 40000
CSV_BENIGN_CAP = 60000
USE_LIVE_FEEDS = True    # pull fresh real phishing/malware URLs (URLhaus + OpenPhish)

build_needed = REBUILD_DATASET or not os.path.exists(DATASET_CSV)

data = None
if not build_needed:
    data = pd.read_csv(DATASET_CSV)
    data = data[data['url'].astype(str).str.len() > 3]
    present = sorted({int(v) for v in data['y'].unique()})
    if present == list(range(len(CLASSES))):
        top_domains, benign_pool = [], pd.Series(dtype=str)
        print(f'Loaded prebuilt dataset from {DATASET_CSV}')
        print('  per-class rows:', {CLASSES[k]: int((data.y == k).sum()) for k in range(len(CLASSES))})
        print('  (set REBUILD_DATASET=True to rebuild from sources)')
    else:
        print(f'Cached dataset has classes {present}, expected {list(range(len(CLASSES)))} '
              f'-> rebuilding from sources.')
        build_needed = True

if build_needed:
    def _fetch(url, timeout=90):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (research)'})
        return urllib.request.urlopen(req, timeout=timeout).read()

    def _fetch_lines(url):
        txt = _fetch(url).decode('utf-8', 'ignore')
        return [ln.strip() for ln in txt.splitlines() if ln.strip() and not ln.startswith('#')]

    # ---- 1. Labelled URLs from the CSV (benign / phishing / malware / defacement) ----
    raw = pd.read_csv(CSV_PATH)
    raw.columns = [c.strip().lower() for c in raw.columns]
    url_col = 'url' if 'url' in raw.columns else raw.columns[0]
    label_col = next((c for c in ['type', 'label', 'result', 'class'] if c in raw.columns),
                     raw.columns[-1])
    lab = raw[label_col].astype(str).str.strip().str.lower()

    def _pick(*names):
        return raw.loc[lab.isin(names), url_col].astype(str).drop_duplicates()

    csv_benign = _pick('benign', 'legitimate', 'legit', 'good', 'safe', '0')
    csv_phish = _pick('phishing', 'phish', '1')
    csv_malware = _pick('malware', 'malicious')
    csv_deface = _pick('defacement', 'deface')
    print(f'From CSV -> benign {len(csv_benign)} | phishing {len(csv_phish)} | '
          f'malware {len(csv_malware)} | defacement {len(csv_deface)}')

    # ---- 1b. Real, RECENT threat feeds (robustness: not just 2021 data) ----
    if USE_LIVE_FEEDS:
        try:
            op = [u for u in _fetch_lines('https://openphish.com/feed.txt') if u.startswith('http')]
            csv_phish = pd.concat([csv_phish, pd.Series(op)]).drop_duplicates()
            print(f'OpenPhish: +{len(op)} recent phishing URLs')
        except Exception as e:
            print('OpenPhish feed skipped:', repr(e)[:100])
        try:
            uh = [u for u in _fetch_lines('https://urlhaus.abuse.ch/downloads/text_online/')
                  if u.startswith('http')]
            csv_malware = pd.concat([csv_malware, pd.Series(uh)]).drop_duplicates()
            print(f'URLhaus: +{len(uh)} recent malware URLs')
        except Exception as e:
            print('URLhaus feed skipped:', repr(e)[:100])

    # ---- 2. Top global domains (popular sites worldwide) ----
    top_domains = []
    try:
        blob = _fetch('http://s3-us-west-1.amazonaws.com/umbrella-static/top-1m.csv.zip')
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            with z.open(z.namelist()[0]) as fh:
                top = pd.read_csv(fh, header=None, names=['rank', 'domain'])
        top_domains = ['https://' + d for d in top['domain'].head(TOP_DOMAINS_N).tolist()]
        print(f'Downloaded {len(top_domains)} top global domains.')
    except Exception as e:
        print('Top-domains download failed:', repr(e)[:120])

    # ---- 3. Worldwide UNIVERSITY domains (teaches the edu/gov pattern globally) ----
    uni_domains = []
    try:
        src = ('https://raw.githubusercontent.com/Hipo/university-domains-list/master/'
               'world_universities_and_domains.json')
        unis = json.loads(_fetch(src).decode('utf-8'))
        for rec in unis:
            for d in rec.get('domains', []):
                if d:
                    uni_domains.append('https://' + d.strip().lower())
        uni_domains = sorted(set(uni_domains))
        print(f'Downloaded {len(uni_domains)} university domains worldwide.')
    except Exception as e:
        print('University-list download failed:', repr(e)[:120])

    # ---- 4. Combine. Benign side = universities + top global + capped generic CSV benign ----
    benign_pool = pd.concat([
        pd.Series(uni_domains),
        pd.Series(top_domains),
        csv_benign.sample(min(CSV_BENIGN_CAP, len(csv_benign)), random_state=42),
    ], ignore_index=True).drop_duplicates()

    data = pd.concat([
        pd.DataFrame({'url': benign_pool, 'y': 0}),      # benign
        pd.DataFrame({'url': csv_phish, 'y': 1}),        # phishing
        pd.DataFrame({'url': csv_malware, 'y': 2}),      # malware
        pd.DataFrame({'url': csv_deface, 'y': 3}),       # defacement
    ], ignore_index=True)
    data = data[data['url'].str.len() > 3]
    # Drop any URL that appears in more than one class (keep first) to avoid label noise.
    data = data.drop_duplicates(subset='url').reset_index(drop=True)

    print('\nPer-class rows:', {CLASSES[k]: int((data.y == k).sum()) for k in range(len(CLASSES))})

## SECTION 2.5: Attack augmentation + benign coverage (the ML-based fix)

A character model alone can't reliably separate `facebook.com` from `facebookk.com`. So we
**teach it by example** — synthesise, from real brand "seeds", both the attacks and the safe
look‑alikes, and let the model learn the boundary:

**Phishing examples generated (label = 1):**

| Family | Example |
|---|---|
| Typosquatting | `facebookk.com`, `paypa1.com`, `amaz0n.com` |
| Combosquatting | `paypal-login.com`, `secure-amazon.net` |
| Subdomain abuse | `paypal.com.secure-update.tk` |

**Benign examples generated (label = 0) — prevents false positives:**
- Every real brand with normal **and** login/secure paths and real subdomains
  (`paypal.com/login`, `accounts.google.com/signin`) → so a *known host* with `/login` is **safe**,
  only a *look‑alike host* is a threat.

**Brand seeds** = top global domains **+ a local Bangladesh list** (banks, MFS, telco, govt). A few
**throwaway brands** (`dropbox`, `reddit`, `pickaboo`) are *held out* of augmentation so Section 7
can prove the model **generalises** to brands it never saw augmented.

Everything is saved into one combined CSV on Drive (auto‑rebuilds if absent), and a **VERIFY**
block prints what's inside.

In [ ]:
# Attack augmentation + HARD-NEGATIVE benign + strong BRAND-BENIGN coverage.
#   Phishing (y=1): typo / combo / subdomain look-alikes of real brands.
#   Benign  (y=0): real brands with normal + login/secure paths + real subdomains, so the
#                  model confidently learns the GENUINE brand domains are safe (counters the
#                  fact that words like "paypal" appear in lots of real phishing).
# HOLDOUT_BRANDS (for the Section 7 generalization test) are throwaway brands, NOT the ones
# we deploy — so protecting real brands never conflicts with the test.
import random
random.seed(42)

AUGMENT_ATTACKS = True
BRAND_N = 6000
HARDNEG_N = 5000
N_TYPO, N_COMBO, N_SUB = 3, 3, 2

GLOBAL_TRUSTED = [
    'google.com', 'microsoft.com', 'github.com', 'apple.com', 'amazon.com', 'facebook.com',
    'meta.com', 'instagram.com', 'whatsapp.com', 'linkedin.com', 'twitter.com', 'x.com',
    'netflix.com', 'youtube.com', 'paypal.com', 'dropbox.com', 'adobe.com', 'yahoo.com',
    'office.com', 'live.com', 'cloudflare.com', 'wikipedia.org', 'reddit.com',
    'stackoverflow.com', 'zoom.us',
]
LOCAL_BRANDS = [
    'bkash.com', 'nagad.com.bd', 'rocket.com.bd', 'upay.com.bd', 'mycash.com.bd',
    'sonalibank.com.bd', 'dutchbanglabank.com', 'bracbank.com', 'islamibankbd.com',
    'citybank.com.bd', 'ebl.com.bd', 'primebank.com.bd', 'pubalibangla.com',
    'agranibank.org', 'rupalibank.org', 'mercantilebank.com.bd', 'ucb.com.bd',
    'abbl.com', 'southeastbank.com.bd',
    'grameenphone.com', 'robi.com.bd', 'banglalink.net', 'gp.com.bd', 'teletalk.com.bd',
    'airtel.com.bd',
    'daraz.com.bd', 'pathao.com', 'foodpanda.com.bd', 'chaldal.com', 'shohoz.com',
    'bdjobs.com', 'rokomari.com', 'pickaboo.com',
    'bb.org.bd', 'bangladesh.gov.bd', 'nbr.gov.bd', 'election.gov.bd', 'passport.gov.bd',
    'prothomalo.com', 'thedailystar.net',
]
SEED_BRANDS = GLOBAL_TRUSTED + LOCAL_BRANDS
# Throwaway brands for the generalization test (NOT deploy-critical). Their look-alikes are
# excluded from training so Section 7 can prove the model still catches them.
HOLDOUT_BRANDS = {'dropbox.com', 'reddit.com', 'pickaboo.com'}

HOMOGLYPHS = {'o': '0', 'l': '1', 'i': '1', 'e': '3', 'a': '4',
              's': '5', 'b': '8', 't': '7', 'g': '9', 'z': '2'}
COMBO_WORDS = ('login', 'secure', 'verify', 'account', 'support', 'update',
               'security', 'signin', 'auth', 'service', 'online', 'help')
COMBO_TLDS = ('com', 'net', 'org', 'info', 'xyz')
ATTACKER_HOSTS = ('secure-update.tk', 'account-verify.cf', 'login-portal.xyz',
                  'web-secure.top', 'signin-alert.online', 'verify-now.click')
SENSITIVE_PATHS = ('login', 'signin', 'account', 'secure/login', 'verify',
                   'auth', 'password/reset', 'user/account', 'settings/security')
NORMAL_PATHS = ('', 'home', 'about', 'products', 'help', 'contact', 'search?q=qr', 'dashboard')
BENIGN_SUBS = ('www', 'accounts', 'login', 'secure', 'my', 'app', 'mail', 'portal')


def _registrable(u):
    h = str(u).split('://')[-1].split('/')[0].lower().strip()
    while h.startswith('www.'):
        h = h[4:]
    return h


def _typo_variants(host):
    name, dot, tld = host.partition('.')
    if not dot or len(name) < 4 or not name.isalnum():
        return []
    out = set()
    for i in range(len(name)):
        out.add(f'{name[:i]}{name[i]}{name[i:]}.{tld}')
        out.add(f'{name[:i]}{name[i + 1:]}.{tld}')
        if name[i] in HOMOGLYPHS:
            out.add(f'{name[:i]}{HOMOGLYPHS[name[i]]}{name[i + 1:]}.{tld}')
    for i in range(len(name) - 1):
        s = list(name); s[i], s[i + 1] = s[i + 1], s[i]
        out.add(f'{"".join(s)}.{tld}')
    for bad in ('tk', 'cf', 'xyz'):
        out.add(f'{name}.{bad}')
    out.discard(host)
    return list(out)


def _combo_variants(host):
    name = host.partition('.')[0]
    if len(name) < 3:
        return []
    out = set()
    for kw in COMBO_WORDS:
        for sep in ('-', ''):
            out.add(f'{name}{sep}{kw}.com')
            out.add(f'{kw}{sep}{name}.com')
        for tld in COMBO_TLDS:
            out.add(f'{name}-{kw}.{tld}')
    return list(out)


def _subdomain_variants(host):
    name = host.partition('.')[0]
    out = set()
    for atk in ATTACKER_HOSTS:
        out.add(f'{host}.{atk}')
        out.add(f'{name}.{atk}')
        out.add(f'{name}-login.{atk}')
    return list(out)


def _attack_examples(host):
    t, c, s = _typo_variants(host), _combo_variants(host), _subdomain_variants(host)
    random.shuffle(t); random.shuffle(c); random.shuffle(s)
    return t[:N_TYPO] + c[:N_COMBO] + s[:N_SUB]


def _hard_negatives(host):
    out = []
    for p in random.sample(SENSITIVE_PATHS, 3):
        out.append(f'https://{host}/{p}')
    for sub in random.sample(BENIGN_SUBS, 3):
        out.append(f'https://{sub}.{host}/{random.choice(SENSITIVE_PATHS)}')
    return out


def _brand_benign(host):
    """Strong benign coverage for a real brand: normal + sensitive paths + subdomains.
    Teaches that the GENUINE registered domain is safe even if its name appears in phishing."""
    out = [f'https://{host}', f'https://www.{host}']
    for p in NORMAL_PATHS:
        out.append(f'https://{host}/{p}' if p else f'https://{host}/')
    for p in SENSITIVE_PATHS:
        out.append(f'https://{host}/{p}')
    for sub in BENIGN_SUBS:
        out.append(f'https://{sub}.{host}/{random.choice(SENSITIVE_PATHS)}')
    return out


if not build_needed:
    attack_urls, benign_extra = [], []
    print('Using prebuilt dataset (already contains attacks + benign coverage) — skipping generation.')
elif AUGMENT_ATTACKS:
    brand_hosts = [_registrable(d) for d in top_domains[:BRAND_N]] + [_registrable(b) for b in SEED_BRANDS]
    benign_hosts = {_registrable(u) for u in benign_pool} | {_registrable(b) for b in SEED_BRANDS}

    # 1) PHISHING attacks (skip held-out throwaway brands).
    seen, attack_urls = set(), []
    for host in brand_hosts:
        if host in HOLDOUT_BRANDS:
            continue
        for v in _attack_examples(host):
            if v not in benign_hosts and v not in seen:
                seen.add(v)
                attack_urls.append('https://' + v)

    # 2) BENIGN coverage: strong for every seed brand + hard-negatives for top hosts.
    seen_b, benign_extra = set(), []
    for b in SEED_BRANDS:                      # strong, deploy-critical brands
        for u in _brand_benign(_registrable(b)):
            if u not in seen_b:
                seen_b.add(u); benign_extra.append(u)
    for d in top_domains[:HARDNEG_N]:          # hard-negatives across many top hosts
        for u in _hard_negatives(_registrable(d)):
            if u not in seen_b:
                seen_b.add(u); benign_extra.append(u)

    data = pd.concat([
        data,
        pd.DataFrame({'url': attack_urls, 'y': 1}),     # attacks = phishing
        pd.DataFrame({'url': benign_extra, 'y': 0}),    # brand-benign + hard negatives
    ], ignore_index=True).drop_duplicates(subset='url')
    print(f'Attacks added: {len(attack_urls)} | benign coverage added: {len(benign_extra)}')
    print('Attack examples:', attack_urls[:4])
    print('Benign examples:', benign_extra[:4])
else:
    attack_urls, benign_extra = [], []
    print('Attack augmentation is OFF (AUGMENT_ATTACKS = False).')

if build_needed:
    data.to_csv(DATASET_CSV, index=False)
    print(f'Saved combined dataset -> {DATASET_CSV}')

# ---- PROOF: read the saved CSV back and show the data is really inside it ----
_chk = pd.read_csv(DATASET_CSV)
_typo = _chk['url'].str.contains('facebookk|paypa1|amaz0n|gooogle|micros0ft|netfllix', case=False, na=False)
_combo = _chk['url'].str.contains('-login|-secure|-verify|-account|-support|-update', case=False, na=False)
_hn = (_chk.y == 0) & _chk['url'].str.contains('/login|/signin|/secure|/account|/verify|/auth', case=False, na=False)
print('\n================ VERIFY qr_training_dataset.csv ================')
print(f'Total rows: {len(_chk)}')
print('Per-class rows:', {CLASSES[k]: int((_chk.y == k).sum()) for k in range(len(CLASSES))})
print(f'  typosquat rows        : {int(_typo.sum())}   e.g. {list(_chk.loc[_typo, "url"].head(2))}')
print(f'  combosquat rows       : {int(_combo.sum())}   e.g. {list(_chk.loc[_combo, "url"].head(2))}')
print(f'  benign w/ login path  : {int(_hn.sum())}   e.g. {list(_chk.loc[_hn, "url"].head(2))}')
print(f'  paypal.com benign rows: {int(((_chk.y==0) & _chk["url"].str.contains("//paypal.com|.paypal.com", regex=True, na=False)).sum())}')
print(f'  sonalibank benign rows: {int(((_chk.y==0) & _chk["url"].str.contains("sonalibank.com.bd", regex=False, na=False)).sum())}')
print('================================================================')

## SECTION 2.6: Dataset analysis (EDA)

A look at the data before modelling — how many URLs per class, the URL‑length distribution by
class, and example URLs. This documents exactly what the model learns from (useful for the thesis).

In [ ]:
import matplotlib.pyplot as plt

EDA_COLORS = ['#16a34a', '#d97706', '#dc2626', '#7c3aed']
counts = [int((data.y == k).sum()) for k in range(len(CLASSES))]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
# (a) examples per class
ax[0].bar(CLASSES, counts, color=EDA_COLORS)
ax[0].set_title('Dataset: examples per class'); ax[0].set_ylabel('number of URLs')
for i, c in enumerate(counts):
    ax[0].text(i, c, f'{c:,}', ha='center', va='bottom', fontsize=9)
# (b) URL-length distribution by class (a real signal the model can use)
for k in range(len(CLASSES)):
    lens = data.loc[data.y == k, 'url'].str.len().clip(upper=150)
    ax[1].hist(lens, bins=40, alpha=0.5, label=CLASSES[k], color=EDA_COLORS[k])
ax[1].set_title('URL length distribution by class'); ax[1].set_xlabel('URL length (chars)')
ax[1].set_ylabel('count'); ax[1].legend()
plt.tight_layout(); plt.show()

print('Examples per class:', dict(zip(CLASSES, counts)))
print('Total URLs:', f'{len(data):,}')
print('Sample URLs per class:')
for k in range(len(CLASSES)):
    sample = list(data.loc[data.y == k, 'url'].head(2))
    print(f'  {CLASSES[k]:11}: {sample}')

## SECTION 3: Sample, split, and weight the classes

We keep **a lot** of data per class (up to `MAX_PER_CLASS`) instead of undersampling everything to
the smallest class — that preserves benign diversity and improves real‑world accuracy. The
remaining class imbalance (malware has fewer samples) is handled with **class weights** during
training (rarer classes get a higher weight), so no class is ignored. Data is split **70 / 15 / 15**
into train / validation / test, **stratified** so every class appears in each split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Use a LOT of data per class (robustness) and let class WEIGHTS handle the remaining
# imbalance — instead of undersampling everything down to the smallest class (which throws
# away most benign diversity and hurts real-world accuracy).
MAX_PER_CLASS = 80000

counts = {k: int((data.y == k).sum()) for k in range(len(CLASSES))}
print('Per-class available:', {CLASSES[k]: counts[k] for k in counts})

balanced = pd.concat(
    [data[data.y == k].sample(min(MAX_PER_CLASS, counts[k]), random_state=42)
     for k in range(len(CLASSES))]
).sample(frac=1, random_state=42)   # shuffle
print('Per-class used:', {CLASSES[k]: int((balanced.y == k).sum()) for k in range(len(CLASSES))})

urls = balanced['url'].astype(str).tolist()
y_all = balanced['y'].to_numpy()

# 70 / 15 / 15, stratified so every class is represented in each split.
urls_train, urls_temp, y_train, y_temp = train_test_split(
    urls, y_all, test_size=0.3, random_state=42, stratify=y_all)
urls_val, urls_test, y_val, y_test = train_test_split(
    urls_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Class weights = (total) / (n_classes * class_count). Rarer classes get a higher weight,
# so the model pays equal attention to malware/defacement despite fewer samples.
cw = compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y_train)
class_weight = {i: float(w) for i, w in enumerate(cw)}
print(f'Train {len(urls_train)} | Val {len(urls_val)} | Test {len(urls_test)}')
print('Class weights:', {CLASSES[i]: round(class_weight[i], 2) for i in class_weight})

## SECTION 4: Tokenize URLs + compute brand-similarity features

Two representations are built for every URL:

**1. Character sequence (for the text model).** `normalize_url` strips scheme / `www.` / case
(identical to the backend `ml_service.py`, so training and serving match), then each character
becomes an id (0 = pad, 1 = unknown), padded to length 200.

**2. Brand‑similarity features (the research fix for typosquatting).** Four numbers describing how
the URL's host relates to a list of known brands:

| Feature | Meaning | Strong signal for |
|---|---|---|
| `brand_or_subdomain` | host *is* a brand or a subdomain of one (`paypal.com`, `accounts.google.com`) | **benign** |
| `min_edit_distance` | edits to the nearest brand (normalized) | typosquat |
| `near_miss` | 1–2 edits from a brand | **phishing** (`facebookk.com`) |
| `homoglyph` | look‑alike chars map back to a brand | **phishing** (`paypa1.com`) |

These are **model inputs** the network *learns* to weigh (not hard rules), and the exact same
computation runs in the backend. This is what lets the model robustly separate a genuine brand
from its impersonation — which a character‑only model cannot do reliably.

In [ ]:
import numpy as np

MAXLEN = 200


def normalize_url(url):
    u = (url or '').strip().lower()
    if '://' in u:
        u = u.split('://', 1)[1]   # drop http:// / https://
    while u.startswith('www.'):
        u = u[4:]                  # drop leading www.
    return u


# Build vocab from TRAIN urls only (no leakage). chars start at id 2.
chars = sorted(set(''.join(normalize_url(u) for u in urls_train)))
char_index = {c: i + 2 for i, c in enumerate(chars)}
VOCAB_SIZE = len(char_index) + 2
print(f'Vocab size = {VOCAB_SIZE}, maxlen = {MAXLEN}')


def encode(url):
    seq = [char_index.get(ch, 1) for ch in normalize_url(url)[:MAXLEN]]
    return seq + [0] * (MAXLEN - len(seq))


# ============================================================================
# BRAND-SIMILARITY FEATURES (the research fix for typosquatting/impersonation).
# 4 numbers describing how the URL's host relates to a known brand:
#   [brand_or_subdomain, min_edit_distance(norm), near_miss(1-2 edits), homoglyph]
#   * brand_or_subdomain = host IS a brand or a SUBDOMAIN of one (paypal.com,
#     accounts.google.com, en.wikipedia.org) -> strong benign signal.
#   * near_miss / homoglyph -> strong typosquat signal (facebookk.com, paypa1.com).
# These are model INPUTS (learned), and the SAME code runs in app/services/ml_service.py.
# ============================================================================
try:
    from rapidfuzz.distance import Levenshtein
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz'])
    from rapidfuzz.distance import Levenshtein

BRAND_REF = sorted({_registrable(b) for b in SEED_BRANDS})   # fixed list, saved with the model
BRAND_SET = set(BRAND_REF)
REV_HOMOGLYPH = {'0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's',
                 '8': 'b', '7': 't', '9': 'g', '2': 'z', '$': 's', '@': 'a'}
NFEAT = 4


def sim_features(url):
    """4 brand-similarity features for one URL. MUST match the backend exactly."""
    host = normalize_url(url).split('/')[0].split(':')[0]
    # 1: host is a brand OR a subdomain of a brand (real brand domains, incl. en.wikipedia.org).
    #    Subdomain-abuse like paypal.com.evil.tk does NOT match (brand isn't the suffix).
    brand_or_sub = 1.0 if any(host == b or host.endswith('.' + b) for b in BRAND_REF) else 0.0
    md = min((Levenshtein.distance(host, b) for b in BRAND_REF), default=99)
    near = 1.0 if 1 <= md <= 2 else 0.0
    norm = ''.join(REV_HOMOGLYPH.get(c, c) for c in host)
    homo = 1.0 if (norm in BRAND_SET and brand_or_sub == 0.0) else 0.0
    return [brand_or_sub, min(md, 5) / 5.0, near, homo]


def featurize(urls):
    return np.array([sim_features(u) for u in urls], dtype=np.float32)


X_train = np.array([encode(u) for u in urls_train], dtype=np.int32)
X_val = np.array([encode(u) for u in urls_val], dtype=np.int32)
X_test = np.array([encode(u) for u in urls_test], dtype=np.int32)
print('Computing brand-similarity features (vs', len(BRAND_REF), 'brands)...')
F_train, F_val, F_test = featurize(urls_train), featurize(urls_val), featurize(urls_test)
print('Encoded shapes:', X_train.shape, X_val.shape, X_test.shape, '| features:', F_train.shape)
print('Mean features by class (brand/sub, edit, near, homo):')
for k in range(len(CLASSES)):
    print(f'  {CLASSES[k]:11}', np.round(F_train[y_train == k].mean(axis=0), 3))

## SECTION 5: Build and train the dual-input model

The model has **two inputs** that are fused, so it learns to combine *what the URL says* with
*how it relates to known brands*:

```
(1) chars ─▶ Embedding ─▶ Conv1D ─▶ Multi-Head Self-Attention ─▶ GlobalMaxPool ─▶ Dense
                                                                                     │
(2) similarity features ─▶ Dense(16) ──────────────────────────────────────────────┤
                                                                                     ▼
                                                       Concatenate ─▶ Dense ─▶ softmax (4 classes)
```

- **Character tower** — a Transformer encoder that learns general URL patterns (bad TLDs, IPs,
  `@`, keywords, combosquats).
- **Similarity MLP** — learns from the 4 brand‑similarity features (Section 4).
- **Fusion** — concatenated and classified into `benign / phishing / malware / defacement`.

Training uses **class weights** (Section 3), `EarlyStopping`, and a fixed seed for
reproducibility. An architecture diagram is rendered below the summary.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.keras.utils.set_random_seed(42)   # reproducible training

# DUAL-INPUT model:
#   (1) character sequence -> Transformer encoder (learns the URL text)
#   (2) brand-similarity features -> small MLP (learns "near a known brand")
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128

# --- (1) character tower ---
char_in = layers.Input(shape=(MAXLEN,), name='chars')
x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(char_in)
x = layers.Conv1D(EMBED_DIM, 5, padding='same', activation='relu')(x)
attn = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=EMBED_DIM)(x, x)
attn = layers.Dropout(0.1)(attn)
x = layers.LayerNormalization(epsilon=1e-6)(x + attn)
ff = layers.Dense(FF_DIM, activation='relu')(x)
ff = layers.Dense(EMBED_DIM)(ff)
ff = layers.Dropout(0.1)(ff)
x = layers.LayerNormalization(epsilon=1e-6)(x + ff)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)

# --- (2) brand-similarity feature MLP ---
feat_in = layers.Input(shape=(NFEAT,), name='sim_features')
f = layers.Dense(16, activation='relu')(feat_in)

# --- fuse and classify ---
z = layers.Concatenate()([x, f])
z = layers.Dropout(0.3)(z)
z = layers.Dense(64, activation='relu')(z)
z = layers.Dropout(0.3)(z)
outputs = layers.Dense(len(CLASSES), activation='softmax')(z)

model = models.Model([char_in, feat_in], outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Architecture diagram for the thesis (skipped gracefully if graphviz is unavailable).
try:
    from IPython.display import Image, display
    tf.keras.utils.plot_model(model, to_file='model_architecture.png',
                              show_shapes=True, show_layer_names=False, dpi=64)
    display(Image('model_architecture.png'))
except Exception as e:
    print('plot_model skipped:', repr(e)[:80])

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-5)

print('Training...')
history = model.fit(
    {'chars': X_train, 'sim_features': F_train}, y_train,
    epochs=15,
    batch_size=128,
    validation_data=({'chars': X_val, 'sim_features': F_val}, y_val),
    class_weight=class_weight,
    callbacks=[early_stopping, reduce_lr],
)
print('Done.')

## SECTION 6: Evaluate on the test set

A full evaluation on data the model never saw, with the figures a thesis needs:

- **Training curves** (train vs validation) — shows it learned and did not overfit.
- **Per‑class report** — precision / recall / F1 for benign / phishing / malware / defacement.
- **Confusion matrix (4×4)** — exactly what is confused with what.
- **Per‑class F1 bar chart**.
- **ROC curve + AUC** for the *safe‑vs‑threat* decision (the real security view).
- **Confidence histogram** — correct vs wrong predictions.
- **Safe‑vs‑Threat metrics** — missed threats (false negatives) and false alarms (false positives).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

test_inputs = {'chars': X_test, 'sim_features': F_test}

# --- 1) Training curves: shows it learned and did NOT overfit ---
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history['loss'], label='train'); ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss per epoch'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(history.history['accuracy'], label='train'); ax[1].plot(history.history['val_accuracy'], label='val')
ax[1].set_title('Accuracy per epoch'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

probs = model.predict(test_inputs, verbose=0)
y_pred = probs.argmax(axis=1)
loss, acc = model.evaluate(test_inputs, y_test, verbose=0)
print(f'Test accuracy: {acc:.4f}  |  Test loss: {loss:.4f}\n')
print('Per-class report (precision / recall / F1):')
print(classification_report(y_test, y_pred, target_names=CLASSES))

# --- 2) Four diagnostic plots for the thesis ---
COLORS = ['#16a34a', '#d97706', '#dc2626', '#7c3aed']
fig, ax = plt.subplots(2, 2, figsize=(13, 10))

cm = confusion_matrix(y_test, y_pred, labels=list(range(len(CLASSES))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0, 0])
ax[0, 0].set_title('Confusion matrix (4 classes)'); ax[0, 0].set_xlabel('Predicted'); ax[0, 0].set_ylabel('True')

f1s = f1_score(y_test, y_pred, average=None, labels=list(range(len(CLASSES))))
ax[0, 1].bar(CLASSES, f1s, color=COLORS); ax[0, 1].set_ylim(0, 1); ax[0, 1].set_title('Per-class F1 score')
for i, v in enumerate(f1s):
    ax[0, 1].text(i, v + 0.01, f'{v:.2f}', ha='center')

yt_bin = (y_test != 0).astype(int)
p_threat = 1.0 - probs[:, 0]
fpr, tpr, _ = roc_curve(yt_bin, p_threat); roc_auc = auc(fpr, tpr)
ax[1, 0].plot(fpr, tpr, color='#dc2626', label=f'AUC = {roc_auc:.3f}'); ax[1, 0].plot([0, 1], [0, 1], '--', color='gray')
ax[1, 0].set_title('ROC curve (safe vs threat)'); ax[1, 0].set_xlabel('False positive rate')
ax[1, 0].set_ylabel('True positive rate'); ax[1, 0].legend(loc='lower right')

conf = probs.max(axis=1)
ax[1, 1].hist(conf[y_pred == y_test], bins=20, alpha=0.6, color='#16a34a', label='correct')
ax[1, 1].hist(conf[y_pred != y_test], bins=20, alpha=0.6, color='#dc2626', label='wrong')
ax[1, 1].set_title('Prediction confidence'); ax[1, 1].set_xlabel('max class probability'); ax[1, 1].legend()
plt.tight_layout(); plt.show()

# --- 3) Security view: SAFE (benign) vs THREAT (anything else) ---
yp_bin = (y_pred != 0).astype(int)
print('Safe-vs-Threat (binary collapse) — the real security metric:')
print(classification_report(yt_bin, yp_bin, target_names=['safe', 'threat']))
miss = int(((yt_bin == 1) & (yp_bin == 0)).sum())
fp = int(((yt_bin == 0) & (yp_bin == 1)).sum())
print(f'Missed threats (false negatives): {miss}/{int(yt_bin.sum())}  |  '
      f'False alarms on safe (false positives): {fp}/{int((yt_bin == 0).sum())}  |  ROC-AUC: {roc_auc:.3f}')

## SECTION 7: Generalization test (real-world + held-out brands)

Tests on URLs NOT in the dataset: real legit sites (want **benign**), clear attacks (want a threat
class), and **held-out brand** look-alikes whose variants were *never augmented* in training —
proving the model learned the *pattern*, not the specific brands. Ends with a single **quantitative
held-out detection rate**.

In [ ]:
def predict(u):
    """Return (class_name, confidence) — uses BOTH the char sequence and similarity features."""
    x = np.array([encode(u)], dtype=np.int32)
    f = np.array([sim_features(u)], dtype=np.float32)
    p = model.predict({'chars': x, 'sim_features': f}, verbose=0)[0]
    idx = int(p.argmax())
    return CLASSES[idx], float(p[idx])

# Real, safe URLs — incl. the tricky "brand + login path" cases and deploy-critical brands.
legit = ['https://www.google.com', 'https://github.com/login', 'https://accounts.google.com/signin',
         'https://en.wikipedia.org/wiki/QR_code', 'https://www.amazon.com/gp/cart',
         'https://paypal.com/signin', 'https://www.paypal.com',
         'https://bkash.com', 'https://www.sonalibank.com.bd', 'https://daraz.com.bd/account',
         'https://facebook.com']
phish = ['http://192.168.0.5@paypal-secure.tk/login/verify', 'http://free-gift-card.tk/claim/password',
         'http://45.137.21.9/secure/signin/confirm', 'http://bit.ly/3xPhish',
         'https://www.appleid-verify.tk/login', 'https://paypa1.com', 'https://facebookk.com',
         'https://amaz0n-login.net', 'https://amazon.com.verify-account.tk', 'https://bkash-update.xyz']

# HELD-OUT brands (dropbox/reddit/pickaboo) were NOT augmented in training.
holdout = ['https://dropboxx.com', 'https://dropbox-login.com', 'https://dropbox.com.secure-verify.tk',
           'https://redditt.com', 'https://reddit-account.xyz',
           'https://pickaboo-login.com', 'https://pickab0o.com']


def show(title, urls, want_benign):
    ok = 0
    print(title)
    for u in urls:
        name, conf = predict(u)
        good = (name == 'benign') if want_benign else (name != 'benign')
        ok += good
        print(f'  {"OK " if good else "!! "}{name:11} {conf * 100:5.1f}%   {u}')
    return ok

okl = show('LEGIT (want benign):', legit, want_benign=True)
okp = show('PHISHING / BAD (want NOT benign):', phish, want_benign=False)
okh = show('HELD-OUT BRAND ATTACKS — not in training (want NOT benign):', holdout, want_benign=False)
print(f'\nGeneralization: legit {okl}/{len(legit)} | phishing {okp}/{len(phish)} | held-out {okh}/{len(holdout)}')

## SECTION 7.5: Ablation study (char-only vs dual-input)

This isolates the contribution of the four brand-similarity features. We train a
**character-only** model (the identical Transformer character tower, with the brand
tower removed) and compare it against the full **dual-input** model on the *same*
test set. The key row is accuracy on the **brand-impersonation subset** (URLs whose
near-miss or homoglyph feature fired) — the typosquatting case the brand tower was
designed for. Run this cell to obtain the numbers for the thesis ablation table.


In [ ]:
# ============================================================================
# ABLATION: does the brand-similarity tower actually help?
# Train a CHAR-ONLY model (identical character tower, no brand features) and
# compare it with the dual-input model on the SAME test set.
# ============================================================================
from sklearn.metrics import f1_score, roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.keras.utils.set_random_seed(42)   # same seed as the dual-input model


def build_char_only():
    """Same character tower as the dual model, but no brand-feature input."""
    char_in = layers.Input(shape=(MAXLEN,), name='chars')
    x = layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM)(char_in)
    x = layers.Conv1D(EMBED_DIM, 5, padding='same', activation='relu')(x)
    attn = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=EMBED_DIM)(x, x)
    attn = layers.Dropout(0.1)(attn)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)
    ff = layers.Dense(FF_DIM, activation='relu')(x)
    ff = layers.Dense(EMBED_DIM)(ff)
    ff = layers.Dropout(0.1)(ff)
    x = layers.LayerNormalization(epsilon=1e-6)(x + ff)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(len(CLASSES), activation='softmax')(x)
    m = models.Model(char_in, out)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


char_model = build_char_only()
print('Training CHAR-ONLY ablation model (same data, same schedule)...')
char_model.fit(
    X_train, y_train, epochs=15, batch_size=128,
    validation_data=(X_val, y_val), class_weight=class_weight,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-5)],
    verbose=2,
)


def metrics_of(probs):
    yp = probs.argmax(axis=1)
    acc = float((yp == y_test).mean())
    mf1 = f1_score(y_test, yp, average='macro')
    auc = roc_auc_score((y_test != 0).astype(int), 1.0 - probs[:, 0])
    return yp, acc, mf1, auc


dual_probs = model.predict({'chars': X_test, 'sim_features': F_test}, verbose=0)
char_probs = char_model.predict(X_test, verbose=0)
dyp, dacc, dmf1, dauc = metrics_of(dual_probs)
cyp, cacc, cmf1, cauc = metrics_of(char_probs)

# brand-impersonation subset: near-miss (col 2) OR homoglyph (col 3) feature fired
imp = (F_test[:, 2] == 1) | (F_test[:, 3] == 1)


def imp_acc(yp):
    return float((yp[imp] == y_test[imp]).mean()) if imp.sum() else float('nan')


print('\n================= ABLATION: Char-only vs Dual-input =================')
print(f'{"Metric":34}{"Char-only":>12}{"Dual-input":>13}')
print(f'{"Overall accuracy":34}{cacc:>12.4f}{dacc:>13.4f}')
print(f'{"Macro F1":34}{cmf1:>12.4f}{dmf1:>13.4f}')
print(f'{"Safe-vs-threat ROC-AUC":34}{cauc:>12.4f}{dauc:>13.4f}')
print(f'{"Acc. on brand-impersonation":34}{imp_acc(cyp):>12.4f}{imp_acc(dyp):>13.4f}'
      f'   (n={int(imp.sum())})')
print('=====================================================================')
print('Expected: the brand-similarity tower lifts the brand-impersonation row the most,')
print('which is exactly the typosquatting / homoglyph case it was engineered to catch.')


## SECTION 8: Export the model + tokenizer (download these 2 files)

Saves two files to Drive and verifies the reloaded model:
- **`phishing_url_model.keras`** — the trained dual‑input model.
- **`url_tokenizer.json`** — everything the backend needs to reproduce inputs **exactly**:
  the character index, `maxlen`, the class order, and the **brand reference list** (so the
  backend computes identical similarity features).

Download **both** into `qr-code-fishing-backend/app/models/ml/`.

In [ ]:
import os, json
import numpy as np
import tensorflow as tf

out_dir = '/content/drive/MyDrive'
model_path = os.path.join(out_dir, 'phishing_url_model.keras')
tok_path = os.path.join(out_dir, 'url_tokenizer.json')

model.save(model_path)
# Save everything the backend needs to reproduce features EXACTLY:
# char_index, maxlen, class order, and the brand reference list.
with open(tok_path, 'w', encoding='utf-8') as f:
    json.dump({'char_index': char_index, 'maxlen': MAXLEN, 'classes': CLASSES,
               'brands': BRAND_REF}, f)
print('Saved model     ->', model_path)
print('Saved tokenizer ->', tok_path, '| classes:', CLASSES, '| brands:', len(BRAND_REF))

m2 = tf.keras.models.load_model(model_path)
probs = m2.predict({'chars': X_test, 'sim_features': F_test}, verbose=0)
pred = probs.argmax(axis=1)
acc = float((pred == y_test).mean())
pben = probs[:, 0]
print(f'Reloaded model test accuracy: {acc:.3f}')
print(f'Mean P(benign): real-benign={pben[y_test == 0].mean():.3f}  attacks={pben[y_test != 0].mean():.3f}')
assert acc > 0.70, f'Accuracy too low ({acc:.3f}); train longer / add data.'
print('OK. Download BOTH files into qr-code-fishing-backend/app/models/ml/')